# 实验一：静态页面与 API 数据采集

**平台：SmartPark DataLab　站点：https://115.159.159.82　学时：4 学时**

本 Notebook 完成两类典型 Web 数据采集：

```text
Device ：动态加载 → 找到 API → JSON    → 分页采集全部设备
Company：静态加载 → HTML → BeautifulSoup → "下一页" → 采集 3 页
```

## 0. 环境准备

统一使用一个 `Session`，站点为 HTTPS，证书正常，无需额外参数。

In [1]:
import json
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://115.159.159.82"

session = requests.Session()
session.headers.update({"User-Agent": "SmartPark-DataLab/1.0 (Lab1)"})

---

# 任务一：判断数据加载方式

## 1.1 Device：动态加载

**判断依据**

1. 打开 `/devices`，"查看网页源代码"：源码里只有空的 `<tbody id="device-rows">` 和"正在加载…"占位，
   `Ctrl + F` 搜索页面上任何一台设备名称（如 `循环泵-000001-001`）**都搜不到**；
2. 说明数据不是随初始 HTML 一起返回的；
3. `F12 → Network → Fetch/XHR`，刷新页面，发现页面由 `/static/js/devices.js` 驱动，
   它调用 `fetch('/api/devices?...')` 拉取数据。

**结论：Device 页面是动态加载，数据来自 Device API。**

**API 信息（记录自 Network）**

| 项目 | 值 |
|---|---|
| Request URL | `https://115.159.159.82/api/devices?page=1&page_size=50` |
| Request Method | `GET` |
| Query 参数 | `page`（页码，从 1 开始）、`page_size`（每页条数，前端固定 50） |
| 其他可选参数 | `company_id`、`device_type`、`status`、`zone`（筛选条件，本实验不传） |
| Response | `application/json` |

## 1.2 Company：静态加载

打开 `/companies` → 右键"查看网页源代码" → `Ctrl + F` 搜索页面上的企业名称
（例如 `临港精工001有限公司`），**能在源代码中直接搜到**，
并且能看到完整的 `<article class="company-card" data-company-id="C000001">` 节点。

**结论：Company 页面是静态加载，数据随 HTML 一起返回，可以直接用 BeautifulSoup 解析。**

## 1.3 用 requests 复现第一页请求

按 Network 中观察到的请求，用 `requests` 复现。

In [2]:
api_url = BASE_URL + "/api/devices"

r = session.get(
    api_url,
    params={
        "page": 1,
        "page_size": 50,
    },
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


读取 JSON 并观察结构。

In [3]:
data = r.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


确认分页信息分别由哪个字段表示：

```text
当前页     → page
每页条数   → page_size
设备总数   → total
本页设备列表 → items
```

In [4]:
print("当前页 page      =", data["page"])
print("每页条数 page_size =", data["page_size"])
print("设备总数 total     =", data["total"])
print("本页列表 items    =", len(data["items"]), "条")

当前页 page      = 1
每页条数 page_size = 50
设备总数 total     = 5000
本页列表 items    = 50 条


查看一条真实设备记录，确认 `device_id`、`company_id`、`device_name`、`device_type`、
`model`、`zone`、`rated_power_kw`、`status` 等字段的含义。

In [5]:
print(json.dumps(data["items"][0], ensure_ascii=False, indent=2))

{
  "device_id": "D00000001",
  "company_id": "C000001",
  "device_name": "循环泵-000001-001",
  "device_type": "pump",
  "model": "SP-PUM-800",
  "zone": "智造一区",
  "install_date": "2021-09-19",
  "rated_power_kw": 71.2,
  "sampling_interval_s": 30,
  "protocol": "mqtt",
  "status": "running",
  "updated_at": "2026-09-01T03:58:00"
}


## 1.4 计算总页数

不要在代码里写死"100 页"，而是由接口返回的 `total` 和 `page_size` 计算。

In [6]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


## 1.5 分页采集全部设备

In [7]:
devices = []

for page in range(1, total_pages + 1):
    r = session.get(
        api_url,
        params={
            "page": page,
            "page_size": page_size,
        },
        timeout=10
    )
    r.raise_for_status()

    page_data = r.json()
    devices.extend(page_data["items"])

print("device count:", len(devices))

device count: 5000


---

# 任务二：采集 Company 静态页面（HTML 解析）

## 2.1 请求 HTML

In [8]:
base_url = BASE_URL
current_url = base_url + "/companies"

r = session.get(current_url, timeout=10)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

# 静态加载验证：企业记录直接出现在返回的 HTML 源码中
print("源码中第一个 company-card 出现在第", r.text.find("company-card"), "个字符处")
print("源码中能搜到企业名称：", "临港精工001有限公司" in r.text)

源码中第一个 company-card 出现在第 1537 个字符处
源码中能搜到企业名称： True


## 2.2 设计企业记录的 CSS Selector

先在 Elements 里找到**一条完整企业记录的最外层节点**：

```html
<article class="company-card" data-company-id="C000001">
  <div class="company-card-top">
    <span class="mono">C000001</span>
    <span class="badge active">运营中</span>
  </div>
  <h2><a href="/companies/C000001">临港精工001有限公司</a></h2>
  <dl>
    <div><dt>所属行业</dt><dd>汽车零部件</dd></div>
    <div><dt>所在区域</dt><dd>智造一区</dd></div>
    <div><dt>员工规模</dt><dd>88 人</dd></div>
    <div><dt>设备数量</dt><dd>6 台</dd></div>
  </dl>
  <a class="detail-link" href="/companies/C000001">查看企业档案 →</a>
</article>
```

按"id → class → data-* → 父子层级 → 复杂位置关系"的顺序考虑：

| 候选写法 | 评价 |
|---|---|
| `article.company-card` | 采用。标签 + 语义 class，短、稳定，与卡片数量一一对应 |
| `div.company-grid > article` | 可用，但依赖外层容器，多一层耦合 |
| `section.company-grid article:nth-of-type(1)` | 不采用：只能取到一条，且带位置关系 |
| 浏览器 Copy selector 得到的 `body > main > section.company-grid > article:nth-child(1)` | 不采用：太长，`nth-child` 对结构变化敏感 |

**最终采用 `article.company-card`**：语义明确、长度最短、不受兄弟节点增减影响。

In [9]:
cards = soup.select("article.company-card")

print("本页企业数：", len(cards))

本页企业数： 20


## 2.3 提取企业字段

先解析第一条记录，再在记录内部提取字段。

行业、区域等信息采用 `dt/dd` 结构，先转换成字典。

In [10]:
card = cards[0]

details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [11]:
# company_id 来自 HTML 属性 data-company-id
company_id = card.get("data-company-id")

# name 来自记录内部 h2 > a 的文本
name = card.select_one("h2 a").get_text(strip=True)

# status 来自记录内部 .badge 的文本
status = card.select_one(".company-card-top .badge").get_text(strip=True)

# 行业 / 区域 / 设备数量来自 dt-dd 字典
industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace("台", "").strip())

print(company_id, name, status, industry, zone, device_count)

C000001 临港精工001有限公司 运营中 汽车零部件 智造一区 6


把上面几步封装成函数，用于逐条解析。

In [12]:
def parse_company_card(card):
    # 从一条 <article class="company-card"> 中提取企业字段
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    return {
        # data-* 属性：比解析可见文本更可靠
        "company_id": card.get("data-company-id"),
        # 文本：记录内部的 <h2><a>
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details.get("所属行业"),
        "zone": details.get("所在区域"),
        # 文本 → 整数：去掉单位"台"
        "device_count": int(details.get("设备数量", "0").replace("台", "").strip()),
        "status": card.select_one(".company-card-top .badge").get_text(strip=True),
        "employee_scale": details.get("员工规模"),
    }


companies = [parse_company_card(c) for c in cards]

print("本页企业数：", len(companies))
print(companies[0])

本页企业数： 20
{'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中', 'employee_scale': '88 人'}


---

# 任务三：根据"下一页"连续采集 3 页 Company

页面底部的分页节点：

```html
<nav class="pagination" aria-label="企业分页">
  <span>上一页</span><b>1</b><a href="/companies?page=2">下一页</a>
</nav>
```

观察结果：

```text
标签     ：<a>（第 1 页时"上一页"是不可点击的 <span>）
class    ：父级 nav.pagination；<a> 自身无 class
href     ："/companies?page=2"（相对路径，需要 urljoin 拼接）
```

设计思路：`nav.pagination` 里只有一个"上一页"和一个"下一页"，
当前页是 `<b>`，不可点击的一侧是 `<span>`，
因此**最后一个 `<a>` 一定是"下一页"** → `nav.pagination a:last-child`。

In [13]:
next_link = soup.select_one("nav.pagination a:last-child")

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

下一页
/companies?page=2


In [14]:
# href 是相对路径，用 urljoin 拼成绝对地址
from urllib.parse import urljoin

next_url = urljoin(current_url, next_link.get("href"))
print(next_url)

https://115.159.159.82/companies?page=2


把"请求当前页 → 解析本页企业 → 找到下一页 href"组合进循环，连续采集 3 页。

In [15]:
companies = []
page_sizes = []
current_url = BASE_URL + "/companies"

for page_no in range(1, 4):
    r = session.get(current_url, timeout=10)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    page_companies = [parse_company_card(c) for c in soup.select("article.company-card")]
    companies.extend(page_companies)
    page_sizes.append(len(page_companies))

    print(f"第 {page_no} 页 {current_url} → {len(page_companies)} 条")

    # 找下一页；没有 <a> 说明已到最后一页
    next_link = soup.select_one("nav.pagination a:last-child")
    if next_link is None:
        print("没有下一页，停止")
        break

    current_url = urljoin(current_url, next_link.get("href"))

print("Company 总数：", len(companies))

第 1 页 https://115.159.159.82/companies → 20 条

第 2 页 https://115.159.159.82/companies?page=2 → 20 条


第 3 页 https://115.159.159.82/companies?page=3 → 20 条
Company 总数： 60


---

# 任务四：验证采集结果

## 4.1 数量与 ID 唯一性

In [16]:
print("companies:", len(companies))
print("unique company_id:", len(set(x["company_id"] for x in companies)))

print("devices:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))

companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000


## 4.2 Company–Device 一致性

根据全部 Device 的 `company_id` 统计每家企业实际设备数，再与 Company 页面上的
`device_count` 比较。

In [17]:
device_counts = Counter(x["company_id"] for x in devices)

mismatch_count = 0

for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]
    actual = device_counts.get(cid, 0)

    if expected != actual:
        mismatch_count += 1
        print("不一致：", cid, "页面显示", expected, "实际", actual)

print("Company-Device 不一致企业数：", mismatch_count)

Company-Device 不一致企业数： 0


## 4.3 结果汇总（报告截图用）

把 5 个关键数字放在同一个单元格里输出，便于一次性截图（）。

In [18]:
# 结果汇总：用于实验报告截图 04_validation.png
print("Company 总数：", len(companies))
print("唯一 company_id 数：", len(set(x["company_id"] for x in companies)))
print("Device 总数：", len(devices))
print("唯一 device_id 数：", len(set(x["device_id"] for x in devices)))
print("Company-Device 不一致企业数：", mismatch_count)

Company 总数： 60
唯一 company_id 数： 60
Device 总数： 5000
唯一 device_id 数： 5000
Company-Device 不一致企业数： 0


---

# 任务五：保存采集结果

验证通过后保存为 JSON（本实验只用 JSON，不引入 Pandas / 数据库 / Parquet）。

In [19]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")
print((data_dir / "devices.json").resolve())
print((data_dir / "companies.json").resolve())

数据已保存到 data/ 目录


D:\数据采集\lab01\data\devices.json
D:\数据采集\lab01\data\companies.json


In [20]:
for path in (data_dir / "devices.json", data_dir / "companies.json"):
    print(path.name, "存在：", path.exists(), "大小：", path.stat().st_size, "字节")

devices.json 存在： True 大小： 1977612 字节
companies.json 存在： True 大小： 14031 字节


---

# 完成前自检

```text
□ 会用"查看网页源代码"判断静态 / 动态加载
□ 能在 Network 中找到 Device API（/api/devices）
□ 能读懂 API 分页结构并自动计算总页数
□ Device 共采集 5000 条，device_id 无重复
□ 能设计简洁、稳定的 Company CSS Selector（article.company-card）
□ 能定位"下一页"并读取 href（nav.pagination a:last-child）
□ Company 连续采集 3 页，共 60 条
□ company_id 无重复
□ Company–Device 数量检查通过（不一致 0 家）
□ 已保存 companies.json 和 devices.json
```